Setup: importing functions, setting google drive path, loading data, and  setting its path


In [28]:
### CITATION
# Used google + claude to research methods and libraries that I am unifmailair with like skimage. Also used it for syntax help
###
import os
import kagglehub
import pandas as pd
import numpy as np
from PIL import Image
from google.colab import drive

#helps process and manipulate images (including helping label/segmnet, removing noise, filling in gaps etc)
from skimage.measure import label
from skimage.filters import threshold_otsu
from scipy.ndimage import binary_fill_holes
from skimage.morphology import remove_small_objects, binary_closing, disk

drive.mount('/content/drive')

# 1. Download BRISC into this session
BASE_DIR = kagglehub.dataset_download("briscdataset/brisc2025")
print("Dataset at:", BASE_DIR)

# 2. Load CSV and update file paths so Python can find the MRI images and their masks
seg_df = pd.read_csv('/content/drive/MyDrive/brain_tumor_mri_BRISC_parsed_datasets/segmentation_test.csv')

OLD_BASE = "/kaggle/input/brisc2025"
for col in ["image_path", "mask_path"]:
    seg_df[col] = seg_df[col].str.replace(OLD_BASE, BASE_DIR, regex=False)

print("Number of images:", len(seg_df))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using Colab cache for faster access to the 'brisc2025' dataset.
Dataset at: /kaggle/input/brisc2025
Number of images: 860


Helpers: load images and build masks

In [35]:
# load tumor mask as True/False (True = tumor), with specks under 20 pixels removed
def load_tumor_mask(mask_path):
    mask = np.array(Image.open(mask_path).convert("L")) > 127 #convert to grayscale and  if over 127 mark as tumor
    return remove_small_objects(mask, connectivity=2, min_size=20) #remove specks of under 20 pixels if not connected either diagonally, upwards, downwards, left or right

# keep only the biggest blob in a True/False image
def largest_component(binary):
    labeled, n = label(binary, connectivity=2, return_num=True)
    if n == 0:
        return binary                                # nothing to keep
    sizes = np.bincount(labeled.ravel())[1:]         # pixel count per blob (skip background)
    return labeled == (np.argmax(sizes) + 1)         # True only on the biggest blob

# build a solid True/False mask of the whole head from the MRI
def load_head_mask(image_path):
    img = np.array(Image.open(image_path).convert("L"))
    head = img > threshold_otsu(img) * 0.3           # low threshold: keep dim brain tissue, drop black background
    head = binary_closing(head, disk(5))             # close small gaps in the head outline
    head = binary_fill_holes(head)                   # fill the inside of the head
    return largest_component(head)                   # keep only the head, drop specks/lines

Compute geometric metrics like size of tumor, number of tumors, location (ex upper right) etc

In [32]:
# METRIC 1: SIZE
def compute_size(tumor, head):
    tumor_area = tumor.sum()                              # number of tumor pixels
    head_area = head.sum()                                # number of head pixels

    if head_area > 0:
        tumor_to_head_ratio = tumor_area / head_area      # how much of the head is tumor
    else:
        tumor_to_head_ratio = np.nan                      # no head found -> can't compute

    return tumor_area, head_area, tumor_to_head_ratio


# METRIC 2: COUNT
def compute_count(tumor):
    blobs, number_of_blobs = label(tumor, connectivity=2, return_num=True)
    return number_of_blobs                                # number of separate tumor pieces


# METRIC 3: LOCATION
def compute_location(tumor):
    # no tumor left -> no location
    if tumor.sum() == 0:
        return np.nan, np.nan

    height, width = tumor.shape
    rows, columns = np.nonzero(tumor)                     # row and column of every tumor pixel

    center_x = columns.mean() / width                     # 0 = left edge, 1 = right edge
    center_y = rows.mean() / height                       # 0 = top edge,  1 = bottom edge
    return center_x, center_y


# ALL METRICS FOR ONE IMAGE
def extract_features(image_path, mask_path):
    tumor = load_tumor_mask(mask_path)
    head = load_head_mask(image_path)

    tumor_area, head_area, tumor_to_head_ratio = compute_size(tumor, head)
    tumor_blob_count = compute_count(tumor)
    tumor_center_x, tumor_center_y = compute_location(tumor)

    # put all the results in one dictionary (one row of the final table)
    results = {
        "tumor_area": tumor_area,
        "head_area": head_area,
        "tumor_to_head_ratio": tumor_to_head_ratio,
        "tumor_blob_count": tumor_blob_count,
        "tumor_center_x": tumor_center_x,
        "tumor_center_y": tumor_center_y,
    }
    return results


# test on the first image before running on everything
test = extract_features(df["image_path"][0], df["mask_path"][0])
print(test)

{'tumor_area': np.int64(1151), 'head_area': np.int64(113703), 'tumor_to_head_ratio': np.float64(0.010122863952578208), 'tumor_blob_count': 1, 'tumor_center_x': np.float64(0.5116508742397915), 'tumor_center_y': np.float64(0.4318900684187663)}


Run across all images and saves the result as a file on Drive

In [34]:
# compute features for every image
all_results = []
for i in range(len(df)):
    image_path = df["image_path"][i]
    mask_path = df["mask_path"][i]

    results = extract_features(image_path, mask_path)
    all_results.append(results)

    # print progress every 100 images
    if i % 100 == 0:
        print("Processed", i, "/", len(df))

# turn the list of results into a table (one row per image)
results_df = pd.DataFrame(all_results)

# add the new feature columns to the original table
features_df = df.copy()
for column in results_df.columns:
    features_df[column] = results_df[column]

# save to Drive so we don't have to recompute after Colab restarts
save_path = '/content/drive/MyDrive/brain_tumor_mri_BRISC_parsed_datasets/segmentation_test_features.csv'
features_df.to_csv(save_path, index=False)
print("Saved to:", save_path)

# quick look at the results
print(features_df["tumor_blob_count"].value_counts())    # how many images have 1, 2, 3... tumor blobs
features_df.describe()

Processed 0 / 860
Processed 100 / 860
Processed 200 / 860
Processed 300 / 860
Processed 400 / 860
Processed 500 / 860
Processed 600 / 860
Processed 700 / 860
Processed 800 / 860
Saved to: /content/drive/MyDrive/brain_tumor_mri_BRISC_parsed_datasets/segmentation_test_features.csv
tumor_blob_count
1    815
2     37
3      6
5      1
4      1
Name: count, dtype: int64


,tumor_area,head_area,tumor_to_head_ratio,tumor_blob_count,tumor_center_x,tumor_center_y
count,860.000000,860.000000,860.000000,860.000000,860.000000,860.000000
mean,4165.430233,139885.506977,0.033792,1.065116,0.500451,0.457554
std,4534.299757,51452.420422,0.032394,0.309631,0.105769,0.124738
min,214.000000,13705.000000,0.001182,1.000000,0.153171,0.124926
25%,1615.750000,111509.750000,0.011072,1.000000,0.438492,0.380491
50%,2877.000000,141034.500000,0.021961,1.000000,0.502169,0.449768
75%,5079.500000,169228.000000,0.046786,1.000000,0.571045,0.552159
max,65890.000000,737280.000000,0.227221,5.000000,0.791600,0.794420
